<a href="https://colab.research.google.com/github/marikaitiprim/Project-DL4Audio/blob/main/DL4AM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model 1: Audio2Midi Melody Transcription

In [ ]:
#insert pretrained model and test

# Model 2: ImprovNet

In [ ]:
import os

# If exists don't clone
if not os.path.exists('/content/improvnet'):
  !git clone https://github.com/keshavbhandari/improvnet.git
  %cd improvnet
else:
  %cd improvnet

In [ ]:
!apt install fluidsynth
!pip install -r requirements.txt

In [ ]:
import gdown

data_url = 'https://drive.google.com/uc?id=11H3y2sFUFldf6nS5pSpk8B-bIDHtFH4K'
data_out = '/content/improvnet/artifacts.zip'
gdown.download(data_url, data_out, quiet=False)

!unzip -q /content/improvnet/artifacts.zip -d /content/improvnet/

In [ ]:
import subprocess

def save_wav(midi_filepath):
    wav_filepath = midi_filepath.replace('.mid', '.wav')
    process = subprocess.Popen(f"fluidsynth soundfont.sf -g 1.0 -r 44100 --quiet --no-shell {midi_filepath} -T wav -F {wav_filepath} > /dev/null", shell=True)
    process.wait()
    return wav_filepath

In [ ]:
# @title Run this to generate the YAML config

import ipywidgets as widgets
from IPython.display import display
import yaml
import subprocess
from collections import OrderedDict

# List of default corruption types for different passes
default_corruption_types = [
    'skyline', 'skyline', 'skyline'
]

default_corruption_rates = [1.0, 1.0, 1.0]

# Define styles to increase the width of the description labels
description_style = {'description_width': '150px'}  # Adjust the width as needed

# Text boxes for generation parameters
convert_from = widgets.Dropdown(options=['jazz', 'classical'], value='classical', description='Convert From:', style=description_style)
convert_to = widgets.Dropdown(options=['jazz', 'classical'], value='jazz', description='Convert To:', style=description_style)
midi_file_path = widgets.Text(value='/content/improvnet/input/Monophonic_Johnny Marks - Rudolph The Red-Nosed Reindeer.mid', description='MIDI Path:', style=description_style)
wav_file_path = widgets.Text(value='', description='WAV Path:', style=description_style)
novel_peaks_pct = widgets.FloatText(value=0.0, description='Preseravtion Ratio:', style=description_style)
temperature = widgets.FloatText(value=1.0, description='Temperature:', style=description_style)
context_before = widgets.IntText(value=5, description='Context Before:', style=description_style)
context_after = widgets.IntText(value=2, description='Context After:', style=description_style)
t_segment_start = widgets.IntText(value=0, description='T Segment Start:', style=description_style)
t_segment_stop = widgets.IntText(value=-1, description='T Segment Stop:', style=description_style)
end_original = widgets.Checkbox(value=True, description='End Original', style=description_style)
use_constraints = widgets.Checkbox(value=True, description='Use Constraints', style=description_style)
reharmonize = widgets.Checkbox(value=False, description='Reharmonize', style=description_style)
write_intermediate_passes = widgets.Checkbox(value=True, description='Write Intermediate Passes', style=description_style)

# UI to input the number of passes
num_passes = widgets.IntText(value=3, description='Number of Passes:', style=description_style)
pass_widgets = []

# Define pass_box before it's used
pass_box = widgets.VBox([])

# Function to create widgets for each pass with different initial corruption types
def update_passes(change):
    global pass_widgets
    pass_widgets = []

    for i in range(change['new']):
        corruption_rate_value = default_corruption_rates[i] if i < len(default_corruption_rates) else 0.5
        # Use BoundedFloatText to limit the range of corruption_rate
        corruption_rate = widgets.BoundedFloatText(
            value=corruption_rate_value,
            min=0.25,
            max=1.0,
            step=0.25,
            description=f'Pass {i+1} Corruption Rate:',
            layout=widgets.Layout(width='250px'),  # Wider layout to prevent text cut-off
            style=description_style
        )

        # Assign different default corruption types based on pass number
        corruption_type_value = default_corruption_types[i] if i < len(default_corruption_types) else 'skyline'

        corruption_type = widgets.Dropdown(
            options=default_corruption_types,
            value=corruption_type_value,
            description=f'Pass {i+1} Corruption Type:',
            layout=widgets.Layout(width='400px'),
            style=description_style)

        pass_widgets.append((corruption_rate, corruption_type))

    # Update the pass_box with the new pass widgets
    pass_box.children = [widgets.HBox(p) for p in pass_widgets]

# Update the number of passes dynamically
num_passes.observe(update_passes, names='value')

# Initial passes
update_passes({'new': num_passes.value})

# Display UI for generation section
display(convert_from, convert_to, midi_file_path, wav_file_path, novel_peaks_pct, temperature, context_before, context_after, t_segment_start, t_segment_stop, end_original, use_constraints, reharmonize, write_intermediate_passes, num_passes)

# Display pass widgets
display(pass_box)

# Button to generate YAML file
def main(b):
    passes = {f'pass_{i+1}': {'corruption_rate': pass_widgets[i][0].value, 'corruption_type': pass_widgets[i][1].value} for i in range(num_passes.value)}

    data = {
        'generation': {
            'convert_from': convert_from.value,
            'convert_to': convert_to.value,
            'midi_file_path': midi_file_path.value,
            'wav_file_path': wav_file_path.value,
            'novel_peaks_pct': novel_peaks_pct.value,
            'temperature': temperature.value,
            'context_before': context_before.value,
            'context_after': context_after.value,
            't_segment_start': t_segment_start.value,
            't_segment_stop': t_segment_stop.value,
            'end_original': end_original.value,
            'use_constraints': use_constraints.value,
            'reharmonize': reharmonize.value,
            'write_intermediate_passes': write_intermediate_passes.value,
            'passes': passes
        },
        'raw_data': {
            'artifact_folder': 'artifacts',
            'eval_folder': 'evaluations'
        },
        'model': {
            'encoder_max_sequence_length': 2048,
            'decoder_max_sequence_length': 512,
            'encoder_num_layers': 12,
            'encoder_num_heads': 8,
            'encoder_hidden_size': 512,
            'encoder_intermediate_size': 2048,
            'decoder_num_layers': 12,
            'decoder_num_heads': 8,
            'decoder_hidden_size': 512,
            'decoder_intermediate_size': 2048
        }
    }

    # Write to a YAML file
    with open('configs.yaml', 'w') as file:
        yaml.dump(data, file, default_flow_style=False)

    print("YAML file created successfully!")

# Button to run code
create_button = widgets.Button(description="Create YAML File")
create_button.on_click(main)

# Display button
display(create_button)

In [ ]:
!python improvnet/generation.py --config configs.yaml

In [ ]:
# @title Enter midi file path for a certain pass from the output folder (ex: /content/improvnet/output/pass_4/generated_debussy-clair-de-lune.mid) and click on Generate Wav. Results may take 2-3 minutes depending on the size of the file.

import IPython.display as ipd
from google.colab import output

midi_file_input = widgets.Text(
    value='',
    placeholder='Enter MIDI file path',
    description='MIDI File:',
    disabled=False
)

def on_button_clicked(b):
    midi_filepath = midi_file_input.value
    if midi_filepath:
        try:
            wav_filepath = save_wav(midi_filepath)
            output.clear()  # Clear previous output
            display(ipd.Audio(wav_filepath))
        except Exception as e:
            print(f"Error: {e}")
    else:
        print("Please enter a MIDI file path.")

button = widgets.Button(description="Generate WAV and Play")
button.on_click(on_button_clicked)

display(midi_file_input, button)